
# Deterministic Skill Routing for ADK Agents — Proof of Concept

**Claim under test:** in a Google Agent Development Kit (ADK) multi-agent
system, *which skill an agent uses for a given job can be decided entirely
outside the agent's own reasoning* — by a plain, deterministic lookup — and
enforced structurally, not just requested via a system prompt. The same
agent uses Skill A for Job 1 and Skill B for Job 2, switching at runtime,
without the agent ever being asked "which skill should you use?"

This matters for regulated / auditable agentic systems: if skill choice is
left to the model's judgement, it is a *probabilistic* behaviour you can
only test statistically. If skill choice is made by code before the model
ever runs, it is a *deterministic* property you can prove and pin.

## Design

| Concept | ADK primitive used | Role |
|---|---|---|
| **Agent identity** | `LlmAgent.name` + a persona instruction | *Who* — stable across jobs (e.g. `CaseWorkerAgent`) |
| **Skill** | an `instruction` fragment + exactly the `tools` it needs | *How* — a swappable capability bundle, not a native ADK class |
| **Router** | a plain Python dict, `job_type -> {agent, skill}` | *Decides* — the ONLY place a skill is chosen. No LLM call happens here. |
| **Binder** | `build_agent_for_job()` | *Enforces* — constructs a fresh `LlmAgent` for one job, wiring in only the routed skill's instruction + tools |
| **Runner** | `InMemoryRunner` | *Executes* — runs the bound agent against the job payload |
| **Audit log** | a list of dict records → `pandas.DataFrame` | *Proves* — records every routing decision so determinism can be checked empirically, not just claimed |

The critical design choice is in the **Binder**: it does not hand the agent
all skills and an instruction saying "pick the right one." It constructs an
agent whose `tools=[...]` list *only contains the routed skill's tool(s)*.
There is nothing else in the agent's context for that call that it could
pick even if it wanted to — the enforcement is structural, not advisory.

## Why a stub model, not live Gemini

This sandbox has no Gemini API key and restricted network access, and the
property under test — *is skill selection deterministic and runtime-
configurable* — is a fact about the routing/binding code, not about model
output quality. So `model=` below is a small `BaseLlm` subclass that always
calls whichever single tool it is given (there is only ever one, by
construction) and then returns the tool's result as the final answer. This
is a supported ADK extension point (`BaseLlm.generate_content_async`), used
here purely for offline reproducibility.

**To run this against a real model:** replace `DeterministicStubLlm()`
anywhere it is constructed with `Gemini(model="gemini-2.5-flash")` (or any
ADK-supported model string) and set `GOOGLE_API_KEY` — no other code
changes. The routing/binding/audit logic is model-agnostic by design.

> **Verify before relying on this in a build:** the exact `google-adk`
> class names, field names, and event/content shapes used below are as of
> `google-adk==2.9.0` (installed against a fresh venv while building this
> notebook). ADK is an actively evolving library — check the current
> `google-adk` release notes / API docs before using this pattern in
> production code.


## 1. Setup

In [1]:

import asyncio
import logging
import re
import warnings
from dataclasses import dataclass
from typing import Callable, AsyncGenerator

import pandas as pd
from google.adk.agents import LlmAgent
from google.adk.models import BaseLlm, LlmRequest, LlmResponse
from google.adk.runners import InMemoryRunner
from google.adk.tools import FunctionTool
from google.genai import types

# Quieten ADK's own telemetry/experimental-feature noise for this notebook
# (token-usage-metadata warnings are expected: the stub model reports none;
# the JSON-schema-for-function-declarations notice is an unrelated,
# already-stable-in-practice experimental flag). Neither bears on the
# routing/binding proof below.
logging.getLogger("google_adk").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module=r"google\.adk.*")

import google.adk
print("google-adk version:", getattr(google.adk, "__version__", "unknown"))


google-adk version: 2.9.0



## 2. The offline deterministic model

Implements ADK's `BaseLlm.generate_content_async`. On the first turn it
calls the (single) tool it has been given with the job's payload; on the
second turn (after the tool result comes back) it returns that result as
the final answer. It refuses to run — via the `assert` — if it is ever
handed more than one tool, which is the negative control for skill
isolation used in §6.


In [2]:

class DeterministicStubLlm(BaseLlm):
    """Offline, reproducible stand-in for a real Gemini model.

    Not part of the concept under test — it exists only so this notebook
    runs deterministically without network access or an API key. Swap for
    `Gemini(model=...)` to exercise this against a live model.
    """

    model: str = "stub-deterministic-v1"

    async def generate_content_async(
        self, llm_request: LlmRequest, stream: bool = False
    ) -> AsyncGenerator[LlmResponse, None]:
        tool_names = list(llm_request.tools_dict.keys())
        assert len(tool_names) == 1, (
            "DeterministicStubLlm expects exactly one tool bound per skill; "
            f"got {tool_names}. If this fires, the binder leaked extra tools "
            "into the agent's context for this job — see §6 (tool isolation)."
        )
        tool_name = tool_names[0]

        last = llm_request.contents[-1]
        if last.parts and last.parts[0].function_response is not None:
            # Second turn: the tool has already run: return its result.
            result = last.parts[0].function_response.response.get("result", "")
            yield LlmResponse(
                content=types.Content(role="model", parts=[types.Part(text=str(result))])
            )
            return

        # First turn: call the one tool available with the job payload.
        user_text = next(
            c.parts[0].text
            for c in llm_request.contents
            if c.role == "user" and c.parts and c.parts[0].text
        )
        yield LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(function_call=types.FunctionCall(
                    name=tool_name, args={"text": user_text}
                ))],
            )
        )



## 3. Skills

A `Skill` is a bundle of exactly what an agent needs to do *one* kind of
job: an instruction fragment and its tool. Skills know nothing about
agents or routing — they are pure, reusable capability units. The
scenario is a regulated financial-services contact-centre back office, to
keep it recognisable without using any real client or account data.


In [3]:

def summarize_complaint(text: str) -> str:
    """Summarise a customer complaint in one neutral sentence for the case file."""
    return f"[complaint-summary] {text.strip()[:80]}"


def extract_case_entities(text: str) -> str:
    """Extract case/account/transaction identifiers mentioned in the message."""
    ids = re.findall(r"\b[A-Z]{2,4}-\d{3,8}\b", text)
    return f"[entity-extraction] ids={ids or 'none found'}"


def check_message_tone(text: str) -> str:
    """Classify the message tone as Neutral, Frustrated, or Urgent."""
    lowered = text.lower()
    if any(w in lowered for w in ("immediately", "urgent", "asap")):
        tone = "Urgent"
    elif any(w in lowered for w in ("angry", "unacceptable", "furious", "terrible")):
        tone = "Frustrated"
    else:
        tone = "Neutral"
    return f"[tone-check] {tone}"


def flag_regulated_terms(text: str) -> str:
    """Flag any regulated/high-risk terms present in the message for compliance review."""
    flagged = [w for w in ("guarantee", "risk-free", "advice") if w in text.lower()]
    return f"[compliance-flag] {flagged or 'none'}"


@dataclass(frozen=True)
class Skill:
    skill_id: str
    instruction: str
    tool_fn: Callable[[str], str]


SKILLS: dict[str, Skill] = {
    "complaint-summary": Skill(
        "complaint-summary",
        "Skill: complaint-summary. Call `summarize_complaint` exactly once "
        "with the customer's message, then return its result verbatim.",
        summarize_complaint,
    ),
    "entity-extraction": Skill(
        "entity-extraction",
        "Skill: entity-extraction. Call `extract_case_entities` exactly once "
        "with the message, then return its result verbatim.",
        extract_case_entities,
    ),
    "tone-check": Skill(
        "tone-check",
        "Skill: tone-check. Call `check_message_tone` exactly once with the "
        "message, then return its result verbatim.",
        check_message_tone,
    ),
    "compliance-flag": Skill(
        "compliance-flag",
        "Skill: compliance-flag. Call `flag_regulated_terms` exactly once "
        "with the message, then return its result verbatim.",
        flag_regulated_terms,
    ),
}
print("Registered skills:", list(SKILLS))


Registered skills: ['complaint-summary', 'entity-extraction', 'tone-check', 'compliance-flag']



## 4. Agents

Agent identities are stable personas — *who* is doing the work, not *how*.
Nothing here mentions any skill; that binding happens per job in §5-6.


In [4]:

BASE_AGENTS: dict[str, str] = {
    "CaseWorkerAgent": (
        "You are CaseWorkerAgent, a first-line case-intake agent for a "
        "regulated financial-services contact centre. You only ever use "
        "the single tool made available to you for this task."
    ),
    "QAAgent": (
        "You are QAAgent, a second-line quality and compliance reviewer. "
        "You only ever use the single tool made available to you for this task."
    ),
}
print("Registered agents:", list(BASE_AGENTS))


Registered agents: ['CaseWorkerAgent', 'QAAgent']



## 5. The deterministic router and the skill binder

`ROUTING_TABLE` is the **only** place a job type resolves to an
(agent, skill) pair, and it is a plain dict lookup — no LLM call, no
agent involved. `build_agent_for_job()` is the **binder**: given the
resolved (agent, skill), it constructs a fresh `LlmAgent` whose
`tools=[...]` contains *only* that skill's tool. A `before_model_callback`
records exactly which tool names were visible to the model on the call
that actually happened, which §6 uses as a negative control.


In [5]:

ROUTING_TABLE: dict[str, dict[str, str]] = {
    "summarise_complaint":    {"agent": "CaseWorkerAgent", "skill": "complaint-summary"},
    "extract_case_entities":  {"agent": "CaseWorkerAgent", "skill": "entity-extraction"},
    "check_tone":             {"agent": "CaseWorkerAgent", "skill": "tone-check"},
    "flag_compliance_terms":  {"agent": "QAAgent",         "skill": "compliance-flag"},
}


def build_agent_for_job(agent_id: str, skill_id: str) -> tuple[LlmAgent, list]:
    """Binds exactly one skill's instruction + tool onto one agent instance.

    A fresh agent is constructed per job (not reused/cached) so each job's
    tool set is provably scoped to that job alone — no state or tool
    bindings can leak from one job to the next.
    """
    persona = BASE_AGENTS[agent_id]
    skill = SKILLS[skill_id]

    captured_tool_sets: list[list[str]] = []

    def _capture_tools_seen_by_model(callback_context, llm_request):
        captured_tool_sets.append(sorted(llm_request.tools_dict.keys()))
        return None  # let the call proceed normally

    agent = LlmAgent(
        name=agent_id,
        model=DeterministicStubLlm(),
        instruction=f"{persona}\n\n{skill.instruction}",
        tools=[FunctionTool(skill.tool_fn)],
        before_model_callback=_capture_tools_seen_by_model,
    )
    return agent, captured_tool_sets



## 6. Orchestrator + audit log

`run_job()` does the lookup, binds, runs via ADK's `InMemoryRunner`, and
appends one audit record per job. The audit log is the evidence: every
claim below ("deterministic", "isolated", "dynamic at runtime") is checked
against it with an `assert`, not just described in prose.


In [6]:

AUDIT_LOG: list[dict] = []


async def run_job(job_id: str, job_type: str, payload: str, routing_table: dict) -> dict:
    route = routing_table[job_type]  # <-- the entire routing "decision"
    agent_id, skill_id = route["agent"], route["skill"]

    agent, captured_tool_sets = build_agent_for_job(agent_id, skill_id)
    runner = InMemoryRunner(agent=agent, app_name="tectonic-adk-skills-poc")
    session = await runner.session_service.create_session(
        app_name="tectonic-adk-skills-poc", user_id="demo-user"
    )

    tool_called, final_text = None, None
    async for event in runner.run_async(
        user_id="demo-user",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=payload)]),
    ):
        for part in (event.content.parts if event.content else []):
            if part.function_call:
                tool_called = part.function_call.name
            if part.text:
                final_text = part.text

    record = {
        "job_id": job_id,
        "job_type": job_type,
        "agent": agent_id,
        "skill": skill_id,
        "tool_called": tool_called,
        "tools_visible_to_model": captured_tool_sets[0] if captured_tool_sets else None,
        "output": final_text,
    }
    AUDIT_LOG.append(record)
    return record



## 7. Proof 1 — one agent, several jobs, several skills, all table-driven

`CaseWorkerAgent` runs three different job types below. Nowhere in this
notebook is `CaseWorkerAgent`'s model ever asked "which skill/tool should
you use" — each call's answer to that question was already fixed by
`ROUTING_TABLE` before the agent was even constructed.


In [7]:

JOBS = [
    {"job_id": "job-001", "job_type": "summarise_complaint",
     "payload": "I am absolutely furious, my transfer under ref ACC-4821 has been stuck for a week."},
    {"job_id": "job-002", "job_type": "extract_case_entities",
     "payload": "Please check case ACC-4821 and also TXN-99231 for me."},
    {"job_id": "job-003", "job_type": "check_tone",
     "payload": "This is unacceptable, I need this sorted immediately."},
    {"job_id": "job-004", "job_type": "flag_compliance_terms",
     "payload": "Our adviser said this investment was risk-free, is that right?"},
]

for job in JOBS:
    await run_job(job["job_id"], job["job_type"], job["payload"], ROUTING_TABLE)

pd.DataFrame(AUDIT_LOG)


,job_id,job_type,agent,skill,tool_called,tools_visible_to_model,output
0,job-001,summarise_complaint,CaseWorkerAgent,complaint-summary,summarize_complaint,[summarize_complaint],"[complaint-summary] I am absolutely furious, m..."
1,job-002,extract_case_entities,CaseWorkerAgent,entity-extraction,extract_case_entities,[extract_case_entities],"[entity-extraction] ids=['ACC-4821', 'TXN-99231']"
2,job-003,check_tone,CaseWorkerAgent,tone-check,check_message_tone,[check_message_tone],[tone-check] Urgent
3,job-004,flag_compliance_terms,QAAgent,compliance-flag,flag_regulated_terms,[flag_regulated_terms],[compliance-flag] ['risk-free']



Two of these four jobs (`job-001`, `job-002`) were routed to the *same*
agent, `CaseWorkerAgent`, but resolved to two *different* skills
(`complaint-summary` vs `entity-extraction`) — driven purely by
`job_type`. That is the core claim, already demonstrated once. The rest of
this notebook stress-tests it.



## 8. Proof 2 — deterministic and reproducible

Run `summarise_complaint` five more times with different payloads. If
routing is truly table-driven (not, say, incidentally stable because the
stub model is simple), every single run must resolve to the same
`(agent, skill, tool_called)` triple regardless of payload content.


In [8]:

payloads = [
    "Second reminder about my complaint, still unresolved.",
    "Formal complaint regarding delayed ISA transfer.",
    "Escalating my complaint to the ombudsman if unresolved.",
    "This is my third complaint email this month.",
    "Closing the loop on my earlier complaint, please confirm status.",
]

for i, payload in enumerate(payloads, start=1):
    await run_job(f"job-det-{i:02d}", "summarise_complaint", payload, ROUTING_TABLE)

det_df = pd.DataFrame(AUDIT_LOG).query("job_type == 'summarise_complaint'")
display(det_df[["job_id", "agent", "skill", "tool_called"]])

routes_seen = det_df[["agent", "skill", "tool_called"]].drop_duplicates()
assert len(routes_seen) == 1, f"routing was NOT deterministic: {routes_seen}"
print(f"\nDeterminism holds across {len(det_df)} runs of 'summarise_complaint': "
      f"always {routes_seen.iloc[0].to_dict()}")


,job_id,agent,skill,tool_called
0,job-001,CaseWorkerAgent,complaint-summary,summarize_complaint
4,job-det-01,CaseWorkerAgent,complaint-summary,summarize_complaint
5,job-det-02,CaseWorkerAgent,complaint-summary,summarize_complaint
6,job-det-03,CaseWorkerAgent,complaint-summary,summarize_complaint
7,job-det-04,CaseWorkerAgent,complaint-summary,summarize_complaint
8,job-det-05,CaseWorkerAgent,complaint-summary,summarize_complaint



Determinism holds across 6 runs of 'summarise_complaint': always {'agent': 'CaseWorkerAgent', 'skill': 'complaint-summary', 'tool_called': 'summarize_complaint'}



## 9. Proof 3 — tool isolation (negative control)

It is not enough that the agent *chose* the right tool — it must have had
*no other tool available to choose from*. This checks the
`tools_visible_to_model` captured by the `before_model_callback` in §5:
for the `entity-extraction` job, the model must have seen
`extract_case_entities` and *only* that — none of the other three skills'
tools were reachable, so there was nothing for the model to get wrong.


In [9]:

entity_job = next(r for r in AUDIT_LOG if r["job_id"] == "job-002")
print("Tools visible to the model for job-002:", entity_job["tools_visible_to_model"])

assert entity_job["tools_visible_to_model"] == ["extract_case_entities"], entity_job
other_skill_tools = {s.tool_fn.__name__ for s in SKILLS.values()} - {"extract_case_entities"}
leaked = other_skill_tools & set(entity_job["tools_visible_to_model"])
assert not leaked, f"skill isolation violated, leaked tools: {leaked}"
print("Isolation holds: none of", sorted(other_skill_tools), "were reachable for this job.")


Tools visible to the model for job-002: ['extract_case_entities']
Isolation holds: none of ['check_message_tone', 'flag_regulated_terms', 'summarize_complaint'] were reachable for this job.



## 10. Proof 4 — dynamic at runtime

Mutate `ROUTING_TABLE` in place — standing in for an ops/config change
(e.g. a policy update read from a config service or database row) — and
re-run the **exact same job_type**. No agent code changed, no model
changed, nothing was redeployed: only the routing data changed, and the
resolved agent+skill changes with it.


In [10]:

print("Before hot-swap:", ROUTING_TABLE["summarise_complaint"])

# Simulate an ops/config change: complaints now get QA/compliance review
# instead of a plain summary.
ROUTING_TABLE["summarise_complaint"] = {"agent": "QAAgent", "skill": "compliance-flag"}
print("After hot-swap: ", ROUTING_TABLE["summarise_complaint"])

before = next(r for r in AUDIT_LOG if r["job_id"] == "job-001")
after = await run_job(
    "job-001-rerun", "summarise_complaint",
    "I am absolutely furious, my transfer under ref ACC-4821 has been stuck for a week.",
    ROUTING_TABLE,
)

print("\nSame job_type, before vs after the routing-table change:")
display(pd.DataFrame([before, after])[["job_id", "job_type", "agent", "skill", "tool_called"]])

assert before["agent"] != after["agent"] or before["skill"] != after["skill"], (
    "hot-swap had no effect — routing did not actually change at runtime"
)
print("\nConfirmed: the same job_type now resolves to a different agent/skill, "
      "purely from the table change above.")

# Restore the original mapping so the rest of this notebook (and a re-run)
# behaves as documented in §7.
ROUTING_TABLE["summarise_complaint"] = {"agent": "CaseWorkerAgent", "skill": "complaint-summary"}


Before hot-swap: {'agent': 'CaseWorkerAgent', 'skill': 'complaint-summary'}
After hot-swap:  {'agent': 'QAAgent', 'skill': 'compliance-flag'}

Same job_type, before vs after the routing-table change:


,job_id,job_type,agent,skill,tool_called
0,job-001,summarise_complaint,CaseWorkerAgent,complaint-summary,summarize_complaint
1,job-001-rerun,summarise_complaint,QAAgent,compliance-flag,flag_regulated_terms



Confirmed: the same job_type now resolves to a different agent/skill, purely from the table change above.



## 11. Generalisation — many-to-many, not one agent/one skill

The full audit log by now already contains more than one skill per agent
*and* more than one agent for the same job_type (before/after the
hot-swap). A pivot confirms the mapping is genuinely many-to-many, not
disguised 1:1 pairing.


In [11]:

full_df = pd.DataFrame(AUDIT_LOG)
pivot = full_df.groupby(["agent", "skill"]).size().rename("jobs_routed").reset_index()
display(pivot)

assert full_df.groupby("agent")["skill"].nunique().max() > 1, (
    "expected at least one agent to have been routed to more than one skill"
)
assert full_df.groupby("job_type")["agent"].nunique().max() > 1, (
    "expected at least one job_type to have resolved to more than one agent "
    "(across the hot-swap in §10)"
)
print("Confirmed many-to-many: at least one agent used >1 skill, and at least "
      "one job_type resolved to >1 agent over time.")


,agent,skill,jobs_routed
0,CaseWorkerAgent,complaint-summary,6
1,CaseWorkerAgent,entity-extraction,1
2,CaseWorkerAgent,tone-check,1
3,QAAgent,compliance-flag,2


Confirmed many-to-many: at least one agent used >1 skill, and at least one job_type resolved to >1 agent over time.



## Part 2 — adding a skill without redeploying the service

**Second claim under test:** an agent can gain a brand-new skill purely by
a change to an on-disk **manifest** — no new Python module reaches the
running process, and the process is never restarted. "No redeploy" is
made concrete and checkable two ways below, not just asserted:

1. `sys.modules` is diffed before and after the new skill is added — if
   it is truly unchanged, no code was imported to create the capability.
2. `os.getpid()` is checked before and after — the same OS process served
   both the old skills and the new one.

## Design for Part 2

The reason no new code is needed is that skills stop being bespoke Python
functions and become **configuration of a small, fixed, already-loaded
library of generic tool primitives**:

| Primitive (already in the running process) | What it's parameterised by |
|---|---|
| `truncate_prefix(text, label, length)` | a label and a length |
| `regex_extract(text, pattern, label)` | a regex pattern and a label |
| `keyword_bucket(text, buckets, default, label)` | an ordered list of (name, keywords) buckets |
| `keyword_flag(text, keywords, label)` | a keyword list and a label |

A **manifest** (`skills_manifest.json`, real file, real disk I/O below —
not a Python literal) lists skills as `{skill_id, job_type, agent,
instruction, primitive, params}`. A `SkillManifestRegistry` reads that
file and, for each entry, builds a tool function as a thin closure over an
*already-imported* primitive plus that entry's `params` — never new source
code, only new data.

This trades expressive power for the "no redeploy" guarantee: a new skill
can only do what some combination of the existing primitives can do. A
skill needing genuinely new logic (not expressible as primitive + params)
is real code and still needs a real deploy — named explicitly in §17
rather than glossed over.


In [12]:

import json
import os
import sys
from pathlib import Path


def truncate_prefix(text: str, label: str, length: int = 80) -> str:
    return f"[{label}] {text.strip()[:length]}"


def regex_extract(text: str, pattern: str, label: str) -> str:
    ids = re.findall(pattern, text)
    return f"[{label}] ids={ids or 'none found'}"


def keyword_bucket(text: str, buckets: list[dict], default: str, label: str) -> str:
    lowered = text.lower()
    for bucket in buckets:
        if any(k in lowered for k in bucket["keywords"]):
            return f"[{label}] {bucket['name']}"
    return f"[{label}] {default}"


def keyword_flag(text: str, keywords: list[str], label: str) -> str:
    lowered = text.lower()
    flagged = [k for k in keywords if k in lowered]
    return f"[{label}] {flagged or 'none'}"


# The complete, fixed set of tool primitives this running process ships
# with. A new skill can only be built from what is already in this dict.
PRIMITIVES = {
    "truncate_prefix": truncate_prefix,
    "regex_extract": regex_extract,
    "keyword_bucket": keyword_bucket,
    "keyword_flag": keyword_flag,
}
print("Loaded primitives (this is all the executable code Part 2 ships with):",
      list(PRIMITIVES))


Loaded primitives (this is all the executable code Part 2 ships with): ['truncate_prefix', 'regex_extract', 'keyword_bucket', 'keyword_flag']



## 13. The manifest — the same four skills, as pure configuration

Re-expressing §3's four bespoke Python functions as manifest entries over
the primitives above, to check the refactor changes nothing observable
before any new skill is added.


In [13]:

MANIFEST_PATH = Path("skills_manifest.json")

V1_MANIFEST = {
    "skills": [
        {
            "skill_id": "complaint-summary",
            "job_type": "summarise_complaint",
            "agent": "CaseWorkerAgent",
            "tool_name": "summarize_complaint",
            "instruction": (
                "Skill: complaint-summary. Call `summarize_complaint` exactly "
                "once with the customer's message, then return its result verbatim."
            ),
            "primitive": "truncate_prefix",
            "params": {"label": "complaint-summary", "length": 80},
        },
        {
            "skill_id": "entity-extraction",
            "job_type": "extract_case_entities",
            "agent": "CaseWorkerAgent",
            "tool_name": "extract_case_entities",
            "instruction": (
                "Skill: entity-extraction. Call `extract_case_entities` exactly "
                "once with the message, then return its result verbatim."
            ),
            "primitive": "regex_extract",
            "params": {"pattern": r"\b[A-Z]{2,4}-\d{3,8}\b", "label": "entity-extraction"},
        },
        {
            "skill_id": "tone-check",
            "job_type": "check_tone",
            "agent": "CaseWorkerAgent",
            "tool_name": "check_message_tone",
            "instruction": (
                "Skill: tone-check. Call `check_message_tone` exactly once with "
                "the message, then return its result verbatim."
            ),
            "primitive": "keyword_bucket",
            "params": {
                "buckets": [
                    {"name": "Urgent", "keywords": ["immediately", "urgent", "asap"]},
                    {"name": "Frustrated", "keywords": ["angry", "unacceptable", "furious", "terrible"]},
                ],
                "default": "Neutral",
                "label": "tone-check",
            },
        },
        {
            "skill_id": "compliance-flag",
            "job_type": "flag_compliance_terms",
            "agent": "QAAgent",
            "tool_name": "flag_regulated_terms",
            "instruction": (
                "Skill: compliance-flag. Call `flag_regulated_terms` exactly "
                "once with the message, then return its result verbatim."
            ),
            "primitive": "keyword_flag",
            "params": {"keywords": ["guarantee", "risk-free", "advice"], "label": "compliance-flag"},
        },
    ]
}

MANIFEST_PATH.write_text(json.dumps(V1_MANIFEST, indent=2))
print(f"Wrote {MANIFEST_PATH.resolve()} ({len(V1_MANIFEST['skills'])} skills)")


Wrote /tmp/claude-0/-home-user-tectonic/00311af2-9916-5ecb-ae96-f39af6195a59/scratchpad/exec_dir/skills_manifest.json (4 skills)



## 14. The manifest-driven registry

`SkillManifestRegistry.reload()` is the entire mechanism: read the file,
and for each entry build a tool that is a closure over an already-loaded
primitive plus that entry's `params`. `reload()` is also what the running
service would call in response to a config-change signal (a file watch, a
webhook, a poll) — not modelled here, since the claim under test is *what
happens once reload runs*, not *what triggers it*.


In [14]:

from dataclasses import dataclass as _dataclass  # local alias, avoids clashing with §3's import


@dataclass(frozen=True)
class ManifestSkill:
    skill_id: str
    job_type: str
    agent: str
    instruction: str
    tool_fn: Callable[[str], str]


class SkillManifestRegistry:
    """Turns an on-disk manifest into callable skills — zero bespoke code
    per skill, only configuration of the fixed `PRIMITIVES` set above."""

    def __init__(self, manifest_path: Path):
        self.manifest_path = manifest_path
        self.skills: dict[str, ManifestSkill] = {}

    def reload(self) -> None:
        manifest = json.loads(self.manifest_path.read_text())
        skills: dict[str, ManifestSkill] = {}
        for entry in manifest["skills"]:
            primitive = PRIMITIVES[entry["primitive"]]
            params = entry["params"]
            tool_name = entry["tool_name"]

            def _tool(text: str, _primitive=primitive, _params=params) -> str:
                return _primitive(text, **_params)

            _tool.__name__ = tool_name
            _tool.__doc__ = entry["instruction"]

            skills[entry["skill_id"]] = ManifestSkill(
                skill_id=entry["skill_id"],
                job_type=entry["job_type"],
                agent=entry["agent"],
                instruction=entry["instruction"],
                tool_fn=_tool,
            )
        self.skills = skills

    def routing_table(self) -> dict[str, dict[str, str]]:
        return {s.job_type: {"agent": s.agent, "skill": s.skill_id} for s in self.skills.values()}


def build_agent_for_manifest_skill(registry: SkillManifestRegistry, agent_id: str, skill_id: str):
    persona = BASE_AGENTS[agent_id]
    skill = registry.skills[skill_id]
    captured_tool_sets: list[list[str]] = []

    def _capture_tools_seen_by_model(callback_context, llm_request):
        captured_tool_sets.append(sorted(llm_request.tools_dict.keys()))
        return None

    agent = LlmAgent(
        name=agent_id,
        model=DeterministicStubLlm(),
        instruction=f"{persona}\n\n{skill.instruction}",
        tools=[FunctionTool(skill.tool_fn)],
        before_model_callback=_capture_tools_seen_by_model,
    )
    return agent, captured_tool_sets


MANIFEST_AUDIT_LOG: list[dict] = []


async def run_manifest_job(job_id: str, job_type: str, payload: str, registry: SkillManifestRegistry) -> dict:
    route = registry.routing_table()[job_type]
    agent_id, skill_id = route["agent"], route["skill"]
    agent, captured_tool_sets = build_agent_for_manifest_skill(registry, agent_id, skill_id)

    runner = InMemoryRunner(agent=agent, app_name="tectonic-adk-skills-poc")
    session = await runner.session_service.create_session(app_name="tectonic-adk-skills-poc", user_id="demo-user")
    tool_called, final_text = None, None
    async for event in runner.run_async(
        user_id="demo-user", session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=payload)]),
    ):
        for part in (event.content.parts if event.content else []):
            if part.function_call:
                tool_called = part.function_call.name
            if part.text:
                final_text = part.text

    record = {
        "job_id": job_id, "job_type": job_type, "agent": agent_id, "skill": skill_id,
        "tool_called": tool_called,
        "tools_visible_to_model": captured_tool_sets[0] if captured_tool_sets else None,
        "output": final_text,
    }
    MANIFEST_AUDIT_LOG.append(record)
    return record


registry = SkillManifestRegistry(MANIFEST_PATH)
registry.reload()
print("Skills loaded from manifest:", list(registry.skills))
print("Routing table derived from manifest:", registry.routing_table())


Skills loaded from manifest: ['complaint-summary', 'entity-extraction', 'tone-check', 'compliance-flag']
Routing table derived from manifest: {'summarise_complaint': {'agent': 'CaseWorkerAgent', 'skill': 'complaint-summary'}, 'extract_case_entities': {'agent': 'CaseWorkerAgent', 'skill': 'entity-extraction'}, 'check_tone': {'agent': 'CaseWorkerAgent', 'skill': 'tone-check'}, 'flag_compliance_terms': {'agent': 'QAAgent', 'skill': 'compliance-flag'}}



## 15. Regression check — manifest-driven skills behave identically

Re-run the same four jobs from §7, now entirely through the manifest path,
and diff the outputs against §7's `AUDIT_LOG`. If the refactor from
bespoke functions to primitive + config changed behaviour, this catches it.


In [15]:

for job in JOBS:
    await run_manifest_job(job["job_id"], job["job_type"], job["payload"], registry)

compare = pd.DataFrame(AUDIT_LOG).set_index("job_id")["output"].rename("part1_output").to_frame()
compare["part2_output"] = pd.DataFrame(MANIFEST_AUDIT_LOG).set_index("job_id")["output"]
compare = compare.loc[[j["job_id"] for j in JOBS]]
display(compare)

mismatches = compare[compare["part1_output"] != compare["part2_output"]]
assert mismatches.empty, f"manifest-driven skills diverged from Part 1:\n{mismatches}"
print("Confirmed: manifest-driven skills reproduce Part 1's outputs exactly — "
      "moving to configuration changed nothing observable.")


,part1_output,part2_output
job_id,,
job-001,"[complaint-summary] I am absolutely furious, m...","[complaint-summary] I am absolutely furious, m..."
job-002,"[entity-extraction] ids=['ACC-4821', 'TXN-99231']","[entity-extraction] ids=['ACC-4821', 'TXN-99231']"
job-003,[tone-check] Urgent,[tone-check] Urgent
job-004,[compliance-flag] ['risk-free'],[compliance-flag] ['risk-free']


Confirmed: manifest-driven skills reproduce Part 1's outputs exactly — moving to configuration changed nothing observable.



## 16. Proof 5 — a brand-new skill, added live, with no redeploy

`QAAgent` currently has one skill (`compliance-flag`). Below, a fifth
manifest entry — `vulnerable-customer-flag`, a plausible Consumer-Duty /
safeguarding check — is appended to `skills_manifest.json` **on disk**,
simulating an ops/config push (a ConfigMap update, a git-synced skills
repo, a database row insert). The only call made afterwards is
`registry.reload()`. Everything is checked, not just narrated:

- the new skill is provably absent before the write and present after;
- `sys.modules` is identical before and after — no new code was imported
  anywhere in the process to create this capability;
- `os.getpid()` is identical — this is the same running process throughout,
  not a restart;
- the new job type actually runs successfully through the same
  `InMemoryRunner`/`LlmAgent` plumbing already in use.


In [16]:

assert "vulnerable-customer-flag" not in registry.skills, "test setup assumption violated"

modules_before = set(sys.modules.keys())
pid_before = os.getpid()

# --- simulate an ops/config push: append a new skill to the manifest file ---
manifest_on_disk = json.loads(MANIFEST_PATH.read_text())
manifest_on_disk["skills"].append({
    "skill_id": "vulnerable-customer-flag",
    "job_type": "flag_vulnerable_customer",
    "agent": "QAAgent",
    "tool_name": "flag_vulnerable_customer",
    "instruction": (
        "Skill: vulnerable-customer-flag. Call `flag_vulnerable_customer` "
        "exactly once with the message, then return its result verbatim."
    ),
    "primitive": "keyword_flag",
    "params": {
        "keywords": ["bereavement", "power of attorney", "health condition", "disability", "carer"],
        "label": "vulnerable-customer-flag",
    },
})
MANIFEST_PATH.write_text(json.dumps(manifest_on_disk, indent=2))
print("Config push written to", MANIFEST_PATH.resolve())

# --- the only code executed in response to that push ---
registry.reload()

assert "vulnerable-customer-flag" in registry.skills
assert "flag_vulnerable_customer" in registry.routing_table()
print("New skill now live in the registry:", registry.skills["vulnerable-customer-flag"])

new_job = await run_manifest_job(
    "job-006", "flag_vulnerable_customer",
    "Customer mentioned a recent bereavement and asked about setting up power of attorney.",
    registry,
)
print("\nNew job ran successfully:", new_job)

modules_after = set(sys.modules.keys())
pid_after = os.getpid()
assert modules_after == modules_before, (
    f"unexpected new modules were imported: {modules_after - modules_before}"
)
assert pid_after == pid_before, "process was restarted — this would not prove 'no redeploy'"

print(f"\nConfirmed: {len(modules_after)} modules loaded, identical before and after "
      f"(no new code imported). PID unchanged ({pid_before}). The new capability came "
      "entirely from the manifest file — no code shipped, no restart.")


Config push written to /tmp/claude-0/-home-user-tectonic/00311af2-9916-5ecb-ae96-f39af6195a59/scratchpad/exec_dir/skills_manifest.json
New skill now live in the registry: ManifestSkill(skill_id='vulnerable-customer-flag', job_type='flag_vulnerable_customer', agent='QAAgent', instruction='Skill: vulnerable-customer-flag. Call `flag_vulnerable_customer` exactly once with the message, then return its result verbatim.', tool_fn=<function SkillManifestRegistry.reload.<locals>._tool at 0x7fb64dff18a0>)

New job ran successfully: {'job_id': 'job-006', 'job_type': 'flag_vulnerable_customer', 'agent': 'QAAgent', 'skill': 'vulnerable-customer-flag', 'tool_called': 'flag_vulnerable_customer', 'tools_visible_to_model': ['flag_vulnerable_customer'], 'output': "[vulnerable-customer-flag] ['bereavement', 'power of attorney']"}

Confirmed: 1867 modules loaded, identical before and after (no new code imported). PID unchanged (432). The new capability came entirely from the manifest file — no code shipp


## 17. Proof 6 — the *same agent* was expanded, not replaced

The ask was specifically that an agent gains more skills without a
redeploy, not that a new agent appears. Confirm `QAAgent` itself now spans
two skills.


In [17]:

qa_skills_before = {"compliance-flag"}  # QAAgent's skill set prior to §16
qa_skills_now = {s.skill_id for s in registry.skills.values() if s.agent == "QAAgent"}
print("QAAgent's skills now:", qa_skills_now)

assert qa_skills_now == qa_skills_before | {"vulnerable-customer-flag"}
print("Confirmed: QAAgent — the very same agent identity used throughout this "
      "notebook — went from 1 skill to 2, purely via the manifest change in §16.")

full_manifest_df = pd.DataFrame(MANIFEST_AUDIT_LOG)
pivot2 = full_manifest_df.groupby(["agent", "skill"]).size().rename("jobs_routed").reset_index()
display(pivot2)


QAAgent's skills now: {'compliance-flag', 'vulnerable-customer-flag'}
Confirmed: QAAgent — the very same agent identity used throughout this notebook — went from 1 skill to 2, purely via the manifest change in §16.


,agent,skill,jobs_routed
0,CaseWorkerAgent,complaint-summary,1
1,CaseWorkerAgent,entity-extraction,1
2,CaseWorkerAgent,tone-check,1
3,QAAgent,compliance-flag,1
4,QAAgent,vulnerable-customer-flag,1



## 18. What this proves, and what it deliberately does not

**Proven, with assertions checked against real audit logs (not just
narrated):**

*Part 1 — deterministic routing:*
- Skill selection is a plain data lookup (`ROUTING_TABLE`), never a
  question put to the model — §7.
- The same agent identity uses different skills for different job types,
  and it is reproducible: identical routing on every repeat, regardless of
  payload — §8.
- Isolation is structural, not just instructional: the tools of every
  unrouted skill are provably absent from what the model can call for a
  given job, captured directly from the `LlmRequest` ADK builds — §9.
- The mapping is genuinely dynamic at runtime: changing `ROUTING_TABLE`
  data alone (no agent/model/code change, no redeploy) changes which
  agent+skill a job type resolves to on the very next call — §10.
- The relationship is many-to-many: one agent, many skills; one job type,
  potentially different agents over time as policy changes — §11.

*Part 2 — skills added without a redeploy:*
- Moving from bespoke Python functions to manifest + primitive
  configuration changes nothing observable — §15.
- A brand-new skill, unknown to the process a moment earlier, becomes
  usable from a manifest file edit and a `reload()` call alone — with
  `sys.modules` and `os.getpid()` identical before and after, i.e. no new
  code was imported and no restart occurred — §16.
- The agent that gained the new skill is the *same* agent identity that
  already had one, not a newly-added agent — §17.

**Deliberately out of scope for this PoC (naming the gap rather than
leaving it silently unhandled):**
- **No live model call.** `DeterministicStubLlm` proves the
  routing/binding/isolation mechanics; it says nothing about a real
  Gemini model's output quality, latency, or cost. Swap to `Gemini(...)`
  (§2) to validate that separately.
- **No persistence for the routing table.** `ROUTING_TABLE` (Part 1) is
  an in-memory dict. A production version needs a versioned,
  access-controlled store (e.g. a routing-policy table with an
  audit/outbox event per change — the pattern this repository's own
  `auditability` and `regulatory-compliance` modules already establish for
  other kinds of decisions) so routing changes are themselves reviewable
  and reversible.
- **One tool per skill.** Real skills will often bundle several related
  tools and few-shot examples; `Skill.tool_fn` / a manifest entry's
  `primitive` would become a list, and the binder's `tools=[...]` would
  fan out over all of them — the binding pattern is unchanged.
- **No "no route found" handling.** An unmapped job type raises
  `KeyError` today; production needs an explicit, logged failure mode
  instead of a silent default.
- **No concurrency/multi-tenant stress test.** Jobs run sequentially,
  each against its own fresh agent/session; concurrent jobs racing a
  registry `reload()` are not exercised.
- **No authorization on who can change routing or the manifest.** In a
  regulated setting, *who* can repoint a job type or add a skill is
  itself a control needing its own audit trail — hot-swapping a dict
  (§10) or a JSON file (§16) stands in for that, not a substitute.
- **No manifest schema validation or provenance check.** §16 trusts the
  manifest's shape and content outright; production needs schema
  validation, and ideally signing/provenance checks, before `reload()`
  ever executes a primitive with attacker- or error-supplied `params`.
- **No reload trigger mechanism.** `reload()` is called explicitly here;
  a real service needs an explicit way to invoke it — a file watcher, a
  webhook, a poll, or an admin endpoint — each with different staleness
  and failure-mode trade-offs, not modelled here.
- **Primitives themselves still need a real deploy.** "No redeploy" holds
  for any skill expressible as an existing primitive + parameters. A
  skill needing genuinely new logic is new code, and that code still
  needs to reach the running process somehow (a real deploy of a new
  primitive, or a dynamic-import plugin mechanism with its own security
  review) — this PoC does not narrow that gap, only names it honestly.
